# 👨‍🍳 Class 2.1: Prompt Engineering, *The Recipe Card*
### GenAI & Agentic AI for SAP Professionals · Day 2 · Morning · 2 hours

> **Today's idea:** the AI is a talented **chef**, and your prompt is the **recipe card**.
> A one-line order gets a random dish. A full recipe card gets **exactly what you wanted**.

| Course | What we cook | Cells |
|---|---|---|
| 🥢 Starter | The empty order: a weak prompt | ▶ 1 |
| 📋 Main 1 | The 5 ingredients of a recipe card: **R-T-C-F-R** | ▶ 2–5 |
| 🧪 Main 2 | Taste test: which ingredient matters most? | ▶ 6 |
| 🍽️ Main 3 | Three cooking styles: zero-shot · few-shot · persona | ▶ 7–10 |
| 🍰 Dessert | Kitchen challenge (hands-on) | ▶ 11–14 |

**Runs locally on Ollama.** The model name lives in `bootcamp_config.json`.
Each slide shows a **▶ CELL** number: switch here and run that cell. 🔒 Mock data only.

## ⚙️ ▶ CELL S: Setup
Checks that Ollama is running and the model is downloaded, then says hello.

In [1]:
# ▶ CELL S · SETUP: connect to the local AI (Ollama). Run once.
import json, re, csv, time
from bootcamp_llm import *          # ask_ai, chat, ask_small, ask_json, show, parse_json, check_setup, last_usage, MODEL
from IPython.display import Markdown, display

DATA = json.load(open("../data/emails.json", encoding="utf-8"))
check_setup()
print("\n🤖", ask_ai("Reply with exactly these words: Connection OK", temperature=0, max_tokens=10),
      f"· {last_usage['seconds']} s · model = {MODEL}")

✅ Ollama 0.40.2 is running at http://localhost:11434
✅ Main model : gemma4:e2b
✅ Small model: gemma4:e2b
⚙️  Config: /Users/ra20586957wipro.com/Documents/Enchanté/Conversations/DC7FB73D-48DB-4ECB-884F-FD8CEBF33240/Day2_Prompt_Engineering_Ollama/bootcamp_config.json



🤖 Connection OK · 28.8 s · model = gemma4:e2b


---
# 🥢 Starter: the empty order
### ▶ CELL 1: "Summarise this email". What could go wrong?
✋ *You're the procurement manager. Can you **act** on this answer?*

In [2]:
# ▶ CELL 1 · The empty order
EMAIL_1 = DATA["EMAIL_1"]
print(EMAIL_1, "\n" + "─" * 70)

weak = ask_ai("Summarise this email.\n\n" + EMAIL_1, temperature=0)
show(weak)
print(f"\n⏱ {last_usage['seconds']} s · {len(weak.split())} words")

From: Rahul Verma, Bharat Logistics
Subject: Invoice INV-77821 - payment pending

Hi team,
We sent invoice INV-77821 for PO 4500012346 last month for ₹7,80,000.
Your AP team says the price is wrong, but we followed the rate card from
our last discussion. The goods were delivered to Plant 1000. This payment is
now 20 days overdue and is affecting our cash flow. Please check urgently
and let us know when we can expect payment, otherwise we may need to put
further deliveries on hold.
Regards,
Rahul 
──────────────────────────────────────────────────────────────────────


Here's a summary of the email:

**Rahul Verma from Bharat Logistics is urgently requesting payment for invoice INV-77821 amounting to ₹7,80,000, which is now 20 days overdue.** The recipient's Accounts Payable team is disputing the price, but Rahul insists they followed the agreed rate card. Rahul warns that the delay is affecting their cash flow and threatens to **halt further deliveries** if payment is not resolved immediately.


⏱ 1.9 s · 67 words


> 🍜 A tasty dish, but not **your** dish: no escalation decision, no owner, no next step.
> **The chef isn't the problem. The order is.**

---
# 📋 Main 1: the recipe card, R-T-C-F-R
| Ingredient | Question | Our example |
|---|---|---|
| 🧑‍🍳 **R**ole | Who is cooking? | SAP procurement analyst |
| 🎯 **T**ask | Which dish? | Summarise the dispute |
| 🧂 **C**ontext | House rules | Above ₹5,00,000 → Finance Controller |
| 🍱 **F**ormat | How to plate it | Fixed fields |
| 🚫 **R**ules | Allergies | No guessing: write "Not stated" |

### ▶ CELL 2: Write the 5 ingredients as separate pieces

In [3]:
# ▶ CELL 2 · The 5 ingredients
ROLE    = "You are an SAP procurement analyst."
TASK    = "Task: Summarise this invoice dispute email."
CONTEXT = ("Context (company policy):\n"
           "- ₹5 lakh = ₹5,00,000.\n"
           "- If the disputed invoice amount is ABOVE ₹5,00,000 → escalate to the Finance Controller.\n"
           "- Otherwise → the AP team handles it.")
FORMAT  = ("Format:\n- Vendor:\n- Invoice number:\n- PO number:\n- Invoice amount:\n- Issue (1 line):\n"
           "- Escalate to Finance Controller? (Yes/No + reason):\n- Next action:")
RULES   = ('Rules:\n- If a value is not in the email, write "Not stated". Do not guess.\n'
           "- Keep each line under 20 words.")

def recipe(*parts, email=EMAIL_1):
    """Stack the chosen ingredients, then add the email between <<< >>> markers."""
    return "\n".join(parts) + "\n\nEmail:\n<<<\n" + email + "\n>>>"

print(recipe(ROLE, TASK, CONTEXT, FORMAT, RULES))

You are an SAP procurement analyst.
Task: Summarise this invoice dispute email.
Context (company policy):
- ₹5 lakh = ₹5,00,000.
- If the disputed invoice amount is ABOVE ₹5,00,000 → escalate to the Finance Controller.
- Otherwise → the AP team handles it.
Format:
- Vendor:
- Invoice number:
- PO number:
- Invoice amount:
- Issue (1 line):
- Escalate to Finance Controller? (Yes/No + reason):
- Next action:
Rules:
- If a value is not in the email, write "Not stated". Do not guess.
- Keep each line under 20 words.

Email:
<<<
From: Rahul Verma, Bharat Logistics
Subject: Invoice INV-77821 - payment pending

Hi team,
We sent invoice INV-77821 for PO 4500012346 last month for ₹7,80,000.
Your AP team says the price is wrong, but we followed the rate card from
our last discussion. The goods were delivered to Plant 1000. This payment is
now 20 days overdue and is affecting our cash flow. Please check urgently
and let us know when we can expect payment, otherwise we may need to put
further deli

### ▶ CELL 3: Add ingredients one at a time
Watch the dish change with each ingredient.

In [4]:
# ▶ CELL 3 · Build the recipe step by step
steps = {
    "1️⃣ Role + Task":                (ROLE, TASK),
    "2️⃣ + Format":                   (ROLE, TASK, FORMAT),
    "3️⃣ + Context + Rules (full)":   (ROLE, TASK, CONTEXT, FORMAT, RULES),
}
for name, parts in steps.items():
    display(Markdown(f"### {name}"))
    show(ask_ai(recipe(*parts), temperature=0))
    print(f"⏱ {last_usage['seconds']} s")

### 1️⃣ Role + Task

## Invoice Dispute Summary for SAP Procurement Analyst

**Subject:** Dispute regarding Invoice INV-77821 (PO 4500012346)

**Sender:** Rahul Verma, Bharat Logistics

**Invoice Details:**
* **Invoice Number:** INV-77821
* **Purchase Order:** PO 4500012346
* **Amount:** ₹7,80,000
* **Goods Delivered To:** Plant 1000

**Issue:**
The vendor (Bharat Logistics) is disputing the invoice amount, claiming the price is incorrect. They assert that they adhered to the **rate card agreed upon during a previous discussion.**

**Urgency/Impact:**
* The payment is **20 days overdue**.
* The delay is negatively impacting the vendor's **cash flow**.
* The vendor has threatened to **suspend future deliveries** if payment is not resolved urgently.

**Action Required (For Procurement/AP Team):**
The AP team needs to urgently review the pricing discrepancy, referencing the agreed-upon rate card, and provide a clear timeline for payment to the vendor.

⏱ 3.8 s


### 2️⃣ + Format

Here is the summary of the invoice dispute email:

- **Vendor:** Bharat Logistics
- **Invoice number:** INV-77821
- **PO number:** 4500012346
- **Invoice amount:** ₹7,80,000
- **Issue (1 line):** Disagreement over the invoiced price, with the vendor citing adherence to the agreed rate card. Payment is 20 days overdue.
- **Escalate to Finance Controller? (Yes/No + reason):** Yes + Due to potential disruption of future deliveries.
- **Next action:** Investigate the price discrepancy and confirm payment timeline.

⏱ 2.3 s


### 3️⃣ + Context + Rules (full)

- Vendor: Bharat Logistics
- Invoice number: INV-77821
- PO number: 4500012346
- Invoice amount: ₹7,80,000
- Issue (1 line): Disputed price based on rate card.
- Escalate to Finance Controller? Yes (Amount > ₹5,00,000)
- Next action: Finance Controller review.

⏱ 1.5 s


### ▶ CELL 4: The scorecard. Which facts made it onto the plate?
A simple, fair check: does the answer contain the facts a manager needs?

In [5]:
# ▶ CELL 4 · Scorecard: weak order vs full recipe card
MUST_HAVE = {"Invoice no.": "INV-77821", "PO no.": "4500012346", "Amount": "7,80,000",
             "Escalation": "Escalate", "Next action": "Next action"}

full = ask_ai(recipe(ROLE, TASK, CONTEXT, FORMAT, RULES), temperature=0)

print(f"{'Fact':14} | {'Weak order':10} | {'Recipe card':11}")
print("─" * 42)
for label, needle in MUST_HAVE.items():
    w = "✅" if needle.lower() in weak.lower() else "❌"
    f = "✅" if needle.lower() in full.lower() else "❌"
    print(f"{label:14} | {w:^10} | {f:^11}")

Fact           | Weak order | Recipe card
──────────────────────────────────────────
Invoice no.    |     ✅      |      ✅     
PO no.         |     ❌      |      ✅     
Amount         |     ✅      |      ✅     
Escalation     |     ❌      |      ✅     
Next action    |     ❌      |      ✅     


### ▶ CELL 5: Kitchen tip, house rules must be **spelled out**
Small local models follow **explicit** rules far better than vague ones. Compare:

In [6]:
# ▶ CELL 5 · Vague context vs explicit context
VAGUE    = "Context: Disputes on invoices above ₹5 lakh must go to the finance controller."
for name, ctx in [("😕 Vague", VAGUE), ("✅ Explicit", CONTEXT)]:
    answer = ask_ai(recipe(ROLE, TASK, ctx, FORMAT, RULES), temperature=0)
    line = next((l for l in answer.splitlines() if "escalat" in l.lower()), "(no escalation line)")
    print(f"{name:11} → {line.strip()}")

😕 Vague     → - Escalate to Finance Controller? No + Price dispute is below ₹5 lakh.


✅ Explicit  → - Escalate to Finance Controller? Yes (Amount > ₹5,00,000)


> 🔑 *Spell out numbers, thresholds and who does what.* "₹5 lakh = ₹5,00,000. Above it → Finance Controller." That's how you'd brief a new joiner, too.

---
# 🧪 Main 2: taste test
### ▶ CELL 6: Remove one ingredient at a time. What breaks?

In [7]:
# ▶ CELL 6 · Leave one ingredient out
ingredients = {"Role": ROLE, "Task": TASK, "Context": CONTEXT, "Format": FORMAT, "Rules": RULES}
for missing in ingredients:
    parts = [v for k, v in ingredients.items() if k != missing]
    answer = ask_ai(recipe(*parts), temperature=0)
    score = sum(needle.lower() in answer.lower() for needle in MUST_HAVE.values())
    print(f"Without {missing:8} → {score}/{len(MUST_HAVE)} facts · {len(answer.split()):>3} words")

Without Role     → 5/5 facts ·  43 words


Without Task     → 5/5 facts ·  37 words


Without Context  → 5/5 facts ·  47 words


Without Format   → 3/5 facts ·  44 words


Without Rules    → 5/5 facts ·  69 words


> 🍱 Usually **Format** is the ingredient you miss most: without it, the plate looks different every time.

---
# 🍽️ Main 3: three cooking styles
### ▶ CELL 7: Zero-shot, just ask

In [8]:
# ▶ CELL 7 · Zero-shot: no examples
question = DATA["CLASSIFY_SET"]["C"]
show(ask_ai(f'Classify this email: "{question}"', temperature=0))

This email can be classified as a **Request for Information (RFI)** or a **Business Inquiry**.

Here's a breakdown of why and other ways it could be categorized depending on the context:

**Primary Classification:**

* **Request for Information (RFI):** This is the most accurate classification. The sender is explicitly asking for specific data (the unit price).
* **Business Inquiry:** It's a standard communication within a business context, seeking necessary commercial data.

**Secondary Classifications (Depending on the relationship and goal):**

* **Procurement/Purchasing Request:** If the sender is actively trying to buy the material, this is a direct step in the purchasing process.
* **Internal Communication:** If this is between colleagues within the same company, it's an internal operational request.
* **Vendor Communication:** If the sender is a client or partner asking a supplier for pricing.

**In summary, the core function of the email is to solicit specific commercial data.**

😕 Its **own** labels and its own format, so a system can't use it.

### ▶ CELL 8: Few-shot, show 3 examples first

In [9]:
# ▶ CELL 8 · Few-shot: 3 examples teach the labels
FEW_SHOT = """Classify each email into exactly one category:
INVOICE_ISSUE, DELIVERY_DELAY, PRICE_QUERY, OTHER.
Reply with the category only.

Email: "The goods for PO 4500012345 haven't arrived yet." → DELIVERY_DELAY
Email: "Invoice 9001 shows a higher price than agreed." → INVOICE_ISSUE
Email: "What is your price for 500 units of bearings?" → PRICE_QUERY

Email: "{email}" →"""

print(ask_ai(FEW_SHOT.format(email=question), temperature=0, max_tokens=10))

PRICE_QUERY


### ▶ CELL 9: The whole tray. Classify 5 emails and score against the answer key

In [10]:
# ▶ CELL 9 · Few-shot on 5 emails, auto-scored
score = 0
for k, email in DATA["CLASSIFY_SET"].items():
    label = ask_ai(FEW_SHOT.format(email=email), temperature=0, max_tokens=10).strip().strip("*.` ")
    ok = label == DATA["CLASSIFY_KEY"][k]
    score += ok
    print(f"{k}: {label:15} {'✅' if ok else '❌ expected ' + DATA['CLASSIFY_KEY'][k]}   {email[:55]}…")
print(f"\n🏆 Score: {score}/5")

A: DELIVERY_DELAY  ✅   The goods for PO 4500012349 were due last week and stil…
B: INVOICE_ISSUE   ✅   Invoice INV-61002 shows ₹520 per unit, but our PO says …


C: PRICE_QUERY     ✅   Could you confirm the current unit price for material M…
D: OTHER           ✅   Please update our bank details for future payments; new…


E: DELIVERY_DELAY  ✅   Our truck broke down; the bearings for PO 4500012347 wi…

🏆 Score: 5/5


> 🚩 **Email D:** "Please update our bank details". It's labelled OTHER, but it's a classic **payment-fraud pattern**. Should an AI ever act on it alone? (Day 4: guardrails.)

### ▶ CELL 10: Persona, same email, three chefs

In [11]:
# ▶ CELL 10 · Three personas, one email
EMAIL_2 = DATA["EMAIL_2"]
personas = {
    "🧑‍💼 Procurement analyst (internal)": "You are an SAP procurement analyst. Summarise for the internal team in 3 bullets.",
    "🤝 Vendor relations (reply)":        "You are a polite vendor relations manager. Draft a short reply to the vendor (max 5 lines).",
    "📊 CFO (one line)":                  "You are writing for a CFO. Give a 1-line update.",
}
for name, instruction in personas.items():
    display(Markdown(f"#### {name}"))
    show(ask_ai(instruction + "\n\n<<<\n" + EMAIL_2 + "\n>>>", temperature=0))

#### 🧑‍💼 Procurement analyst (internal)

Here is the summary for the internal team:

* **Invoice Blocked:** Invoice INV-55190 for PO 4500012348 (amounting to ₹12,00,000) is currently blocked.
* **Reason for Block:** The block is due to a price variance, specifically related to the steel surcharge communicated in August.
* **Action Required:** Procurement needs to confirm the revised price to lift the block on the invoice.

#### 🤝 Vendor relations (reply)

Dear Lakshmi,

Thank you for reaching out regarding PO 4500012348.

I will immediately check with our procurement team regarding the price variance and the steel surcharge communication.

We aim to resolve this swiftly. I will update you shortly.

Best regards,
[Your Name]

#### 📊 CFO (one line)

**Invoice for Gearbox order is blocked pending procurement confirmation on the revised price due to steel surcharge.**

---
# 🍰 Dessert: the Kitchen Challenge (40 min)
Change only the lines marked **← change me**, then press **Shift + Enter**.

### ▶ CELL 11: Challenge 1, write your own recipe card for Email 3

In [12]:
# ▶ CELL 11 · Your recipe card
EMAIL_3 = DATA["EMAIL_3"]
print(EMAIL_3, "\n" + "─" * 70)

my_role    = "You are an SAP accounts payable clerk."                               # ← change me
my_task    = "Task: Summarise this email and recommend an action."                  # ← change me
my_context = "Context: Quantity differences of more than 2% need a credit note."    # ← change me
my_format  = "Format:\n- Vendor:\n- Invoice:\n- Difference:\n- Credit note needed (Yes/No):\n- Next action:"   # ← change me
my_rules   = 'Rules: If a value is missing, write "Not stated". Max 15 words per line.'  # ← change me

show(ask_ai(recipe(my_role, my_task, my_context, my_format, my_rules, email=EMAIL_3), temperature=0))

From: Arjun Mehta, Delta Plastics
Subject: Small invoice query

Hi,
Invoice INV-30117 for PO 4500012353 (₹1,20,000) shows 400 housings, but our
delivery note says 380 were accepted at Plant 2000. Shall we issue a credit note
for the 20 rejected pieces?
Regards, Arjun 
──────────────────────────────────────────────────────────────────────


- Vendor: Delta Plastics
- Invoice: INV-30117
- Difference: 20 housings (approx 1.67%)
- Credit note needed (Yes): Yes
- Next action: Issue credit note for 20 rejected pieces.

✅ *Check with Python:* 20 of 400 pieces = **5%** → above 2% → credit note **Yes**.
> 🧮 In our test run the model wrote *"approx 1.67%"*: **wrong maths, right decision**. Never trust AI arithmetic (more in Class 2.2).

### ▶ CELL 12: Challenge 2, invent your own labels (few-shot)

In [13]:
# ▶ CELL 12 · Your own few-shot classifier
MY_FEW_SHOT = """Classify the ticket as: MASTER_DATA, AUTHORISATION, PERFORMANCE, OTHER. Reply with the label only.
Ticket: "Please create vendor V500 in company code 1000." → MASTER_DATA
Ticket: "I get 'No authorisation for transaction ME21N'." → AUTHORISATION
Ticket: "{t}" →"""                                                              # ← change the labels/examples

my_tickets = ["Report ZMM_STOCK takes 20 minutes to run.",                      # ← change me
              "Need access to MIGO for plant 2000.",
              "Change the bank key for vendor V300."]
for t in my_tickets:
    print(f"{ask_ai(MY_FEW_SHOT.format(t=t), temperature=0, max_tokens=10).strip():15} ← {t}")

PERFORMANCE     ← Report ZMM_STOCK takes 20 minutes to run.
AUTHORISATION   ← Need access to MIGO for plant 2000.


MASTER_DATA     ← Change the bank key for vendor V300.


### ▶ CELL 13: Challenge 3, the persona switch

In [14]:
# ▶ CELL 13 · Your persona
my_persona = "You are a strict internal auditor. List 3 risks in this email, one line each."   # ← change me
show(ask_ai(my_persona + "\n\n<<<\n" + EMAIL_2 + "\n>>>", temperature=0))

Here are 3 risks identified in the email:

1. **Invoice payment blockage:** The invoice is currently blocked, risking delayed payment.
2. **Unconfirmed price variance:** The justification for the price increase is not formally confirmed, risking financial discrepancy.
3. **Lack of clear escalation path:** The request relies on an unspecified party (procurement) for resolution, risking process inefficiency.

### ▶ CELL 14: Challenge 4, the head chef reviews your recipe card

In [15]:
# ▶ CELL 14 · AI scores your prompt (0–2 per ingredient)
prompt_to_review = "Summarise this email."                                       # ← paste your prompt here
show(ask_ai(f"""Score this prompt 0-2 on each of: Role, Task, Context, Format, Rules (max 10).
Show a small markdown table, then give ONE concrete improvement.
Prompt:
<<<
{prompt_to_review}
>>>""", temperature=0))

Here is the scoring and feedback for the prompt: "Summarise this email."

| Criterion | Score (0-2) | Justification |
| :--- | :--- | :--- |
| **Role** | 1 | No specified role for the summarizer (e.g., executive assistant, journalist). |
| **Task** | 2 | The task is clear: "Summarise." |
| **Context** | 0 | Missing the crucial element: *What* email needs summarizing? (The email itself is missing.) |
| **Format** | 1 | No required output format specified (e. e.g., bullet points, paragraph, length). |
| **Rules** | 1 | No constraints provided (e. e.g., tone, length limit, focus areas). |

**Concrete Improvement:**

Specify the input and the desired output format.

**Improved Prompt Example:**

"Act as a professional assistant. Summarise the following email in three bullet points, focusing only on the action items required. [Insert Email Here]"

---
## 🧾 The bill: Class 2.1 takeaways
1. A prompt is a **recipe card**: Role · Task · Context · Format · Rules.
2. **Spell out** house rules (numbers, thresholds, owners), especially for small local models.
3. **Few-shot** examples teach labels fast. **Personas** change tone and depth.

> 🌉 **After lunch, Class 2.2:** a good dish isn't enough when a **machine** has to read the output, or when the email itself tries to give the chef orders. 🔐

---
## 🛠️ Trainer troubleshooting (Ollama)
| Message | Cause | Fix |
|---|---|---|
| `Cannot reach Ollama at http://localhost:11434` | Ollama isn't running | Windows: Start menu → **Ollama**. Mac: open the Ollama app. Or in a terminal: `ollama serve` |
| `Model '…' is not downloaded` | The model hasn't been pulled | In a terminal: `ollama pull <model name>` |
| `ModuleNotFoundError: bootcamp_llm` | The notebook was opened from a different folder | Open the **whole bundle folder** in VS Code or Antigravity, and keep the notebook inside `notebooks/` |
| Very slow first answer | The model is loading into memory | Normal. The next answers are faster. |
| Slow on every answer | The laptop is too small for the model | Switch `model` in `bootcamp_config.json` to a smaller one (e.g. `llama3.2:1b`), then **Restart kernel** |
| Answers differ from the slides | A different model or a new run | Expected: small local models vary. Discuss *why*. |
| Change model mid-class | | Edit `bootcamp_config.json` → **Restart kernel** → run CELL S again |